# eval_rag.ipynb — RAG 답변 품질 평가

## 평가 설계

```
domain_qna (Qdrant)  →  질문을 쿼리로 RAG 실행  →  2가지 품질 측정

① Context Recall     : 정답이 retrieved context 안에 들어있는가?
② Answer Faithfulness: LLM 응답이 hallucination 없이 context에 근거하는가?
```

gold label = domain_qna 페이로드의 `answer` 필드 (별도 수집 불필요)

| 셀 | 목적 |
|---|---|
| 0 | Qdrant → `data/rag_eval.json` 추출 |
| 1 | 데이터셋 통계 확인 |
| 2 | Context Recall 평가 (키워드 매칭) |
| 3 | Answer Faithfulness 평가 (LLM-as-Judge) |
| 4 | 카테고리별 성능 분석 |
| 5 | 오답 케이스 분석 |

## 0. Qdrant → `data/rag_eval.json` 추출

`domain_qna` 컬렉션을 전체 스캔해서 `(question, answer, category, species)` 저장

In [1]:
import sys, json, os
from pathlib import Path
from dotenv import load_dotenv

HERE = Path(".").resolve()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

load_dotenv()

from qdrant_client import QdrantClient

QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
qc = QdrantClient(url=QDRANT_URL)

# 전체 스캔 (scroll)
records = []
offset  = None

while True:
    batch, next_offset = qc.scroll(
        collection_name="domain_qna",
        limit=200,
        offset=offset,
        with_payload=True,
        with_vectors=False,
    )
    for p in batch:
        pl = p.payload
        q  = (pl.get("question") or "").strip()
        a  = (pl.get("answer")   or "").strip()
        if q and a:
            records.append({
                "id":       str(p.id),
                "question": q,
                "answer":   a,
                "category": pl.get("category", ""),
                "species":  pl.get("species",  ""),
            })
    if next_offset is None:
        break
    offset = next_offset

OUT = HERE / "data" / "rag_eval.json"
with open(OUT, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f"추출 완료: {len(records)}개 → {OUT}")

/opt/anaconda3/envs/final-project/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


추출 완료: 2411개 → /Users/leemdo/Workspaces/SKN22-Final-2Team-WEB/notebooks/langgraph/data/rag_eval.json


## 1. 데이터셋 통계

In [2]:
from collections import Counter
import pandas as pd

with open(HERE / "data" / "rag_eval.json", encoding="utf-8") as f:
    eval_data = json.load(f)

print(f"총 {len(eval_data)}개\n")

cat_cnt  = Counter(r["category"] for r in eval_data)
spec_cnt = Counter(r["species"]  for r in eval_data)

print("[ 카테고리 분포 ]")
for k, v in cat_cnt.most_common():
    print(f"  {k:20s}: {v:4d}개")

print("\n[ species 분포 ]")
for k, v in spec_cnt.most_common():
    print(f"  {k:10s}: {v:4d}개")

총 2411개

[ 카테고리 분포 ]
  사육 및 관리             :  710개
  건강 및 질병             :  576개
  영양 및 식단             :  563개
  행동 및 심리             :  554개
  여행 및 이동             :    8개

[ species 분포 ]
  dog       : 1129개
  cat       : 1115개
  both      :  167개


In [3]:
# 샘플 확인
for r in eval_data[:3]:
    print(f"[{r['category']} / {r['species']}]")
    print(f"  Q: {r['question']}")
    print(f"  A: {r['answer'][:80]}...")
    print()

[건강 및 질병 / cat]
  Q: 고양이 '췌장염'이 재발하기 쉬운 이유는 무엇인가요?
  A: 고양이 췌장염은 원인이 불분명한 경우가 많고, 염증성 장질환(IBD)이나 간 질환이 동반되는 '삼중계염' 형태가 흔하기 때문입니다. 식단 변화나...

[사육 및 관리 / cat]
  Q: 췌장염을 앓았던 고양이의 식단 관리 원칙은?
  A: 췌장에 무리를 주지 않도록 고단백, 저지방, 소화 흡수가 용이한 사료를 선택해야 합니다. 한 번에 많이 먹기보다 소량씩 자주 급여하여 위장관의 ...

[건강 및 질병 / cat]
  Q: 고양이 '당뇨병'의 대표적인 초기 증상 '3다(多)'는?
  A: 다음(물을 많이 마심), 다뇨(소변량이 늘어남), 다식(많이 먹지만 살은 빠짐)입니다. 인슐린 부족으로 세포가 당을 에너지로 쓰지 못해 신체 시...



## 2. Context Recall 평가

```
Context Recall = gold answer의 핵심 키워드가 retrieved contexts에 몇 개나 등장하는가
```

- 키워드 추출: LLM으로 `answer`에서 핵심 명사 3~5개 추출
- 매칭: retrieved contexts 전체 텍스트에서 키워드 포함 여부 확인
- 비용 절감: 전체 대신 카테고리별 샘플 30개씩 평가

In [4]:
import random
from openai import OpenAI

llm = OpenAI()
LLM_MODEL = "gpt-4o-mini"

# 카테고리별 최대 N개 샘플링
N_PER_CAT = 10
random.seed(42)

by_cat: dict[str, list] = {}
for r in eval_data:
    by_cat.setdefault(r["category"], []).append(r)

sample_set = []
for cat, items in by_cat.items():
    sample_set.extend(random.sample(items, min(N_PER_CAT, len(items))))

print(f"평가 샘플: {len(sample_set)}개")
for cat, items in by_cat.items():
    n = sum(1 for r in sample_set if r["category"] == cat)
    print(f"  {cat}: {n}개")

평가 샘플: 48개
  건강 및 질병: 10개
  사육 및 관리: 10개
  행동 및 심리: 10개
  영양 및 식단: 10개
  여행 및 이동: 8개


In [5]:
from utils import hybrid_search
from qdrant_client.models import FieldCondition, Filter, MatchAny, MatchValue

DOMAIN_INTENT_TO_CATEGORY = {
    "health_disease":      "건강 및 질병",
    "care_management":     "사육 및 관리",
    "nutrition_diet":      "영양 및 식단",
    "behavior_psychology": "행동 및 심리",
    "travel":              "여행 및 이동",
}
CAT_TO_DOMAIN = {v: k for k, v in DOMAIN_INTENT_TO_CATEGORY.items()}


def extract_keywords(answer: str) -> list[str]:
    """gold answer에서 핵심 명사 키워드 추출 (LLM)"""
    res = llm.chat.completions.create(
        model=LLM_MODEL,
        messages=[{
            "role": "user",
            "content": (
                f"다음 문장에서 핵심 명사·용어 3~5개를 쉼표로만 반환하세요. 설명 없이.\n{answer}"
            ),
        }],
        temperature=0,
    ).choices[0].message.content
    return [k.strip() for k in res.split(",") if k.strip()]


def recall_score(keywords: list[str], contexts: list[str]) -> float:
    """키워드 중 contexts에 등장하는 비율"""
    if not keywords:
        return 0.0
    full_text = " ".join(contexts).lower()
    hits = sum(1 for kw in keywords if kw.lower() in full_text)
    return hits / len(keywords)


print("Context Recall 평가 시작...")
results = []

for i, item in enumerate(sample_set):
    # RAG 실행
    species  = item["species"]
    category = item["category"]
    must = []
    if species:
        must.append(FieldCondition(key="species", match=MatchAny(any=[species, "both"])))
    if category:
        must.append(FieldCondition(key="category", match=MatchValue(value=category)))
    f = Filter(must=must) if must else None

    points   = hybrid_search("domain_qna", item["question"], top_k=5, qdrant_filter=f)
    contexts = [
        f"{p.payload.get('question','')} {p.payload.get('answer','')}"
        for p in points
    ]

    # 키워드 기반 recall
    keywords = extract_keywords(item["answer"])
    score    = recall_score(keywords, contexts)

    results.append({
        **item,
        "keywords":         keywords,
        "n_contexts":       len(contexts),
        "context_recall":   round(score, 3),
        "contexts":         contexts,
    })
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(sample_set)} 완료")

avg_recall = sum(r["context_recall"] for r in results) / len(results)
print(f"\n전체 Context Recall: {avg_recall:.3f}")

Context Recall 평가 시작...
Dense 모델 로드 중...


/Users/leemdo/Workspaces/SKN22-Final-2Team-WEB/notebooks/langgraph/utils.py:32: UserWarning: The model intfloat/multilingual-e5-large now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  _dense_model  = TextEmbedding("intfloat/multilingual-e5-large")


Sparse 모델 로드 중...
모델 로드 완료
  10/48 완료
  20/48 완료
  30/48 완료
  40/48 완료

전체 Context Recall: 0.981


In [6]:
# 카테고리별 Context Recall
from collections import defaultdict

cat_recall: dict[str, list] = defaultdict(list)
for r in results:
    cat_recall[r["category"]].append(r["context_recall"])

print("[ 카테고리별 Context Recall ]\n")
rows = []
for cat, scores in sorted(cat_recall.items()):
    avg = sum(scores) / len(scores)
    rows.append({"카테고리": cat, "샘플수": len(scores), "Recall": round(avg, 3)})
    print(f"  {cat:20s}: {avg:.3f}  (n={len(scores)})")

print(f"\n  {'전체':20s}: {avg_recall:.3f}")

[ 카테고리별 Context Recall ]

  건강 및 질병             : 0.980  (n=10)
  사육 및 관리             : 0.950  (n=10)
  여행 및 이동             : 1.000  (n=8)
  영양 및 식단             : 1.000  (n=10)
  행동 및 심리             : 0.980  (n=10)

  전체                  : 0.981


## 3. Answer Faithfulness 평가 (LLM-as-Judge)

```
Faithfulness = LLM 응답이 retrieved context 밖의 내용을 생성했는가?
             (hallucination 탐지)

Judge 프롬프트: context + question → LLM 응답 생성 → Judge가 0~1 점수 부여
```

In [7]:
RESPOND_SYSTEM = """당신은 반려동물 전문 AI 어시스턴트입니다. 주어진 참고 지식만 바탕으로 답변하세요."""

JUDGE_PROMPT = """\
다음은 AI 어시스턴트의 답변과 그 근거로 사용된 참고 지식입니다.

[참고 지식]
{context}

[AI 답변]
{response}

[판단 기준]
AI 답변의 모든 주요 주장이 참고 지식에 근거하면 1.0, 일부만 근거하면 0.5, 대부분 근거 없으면 0.0.

0.0 / 0.5 / 1.0 중 하나의 숫자만 반환하세요."""


def generate_answer(question: str, contexts: list[str]) -> str:
    context_text = "\n\n---\n\n".join(contexts[:3])
    return llm.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": RESPOND_SYSTEM},
            {"role": "user",   "content": f"[참고 지식]\n{context_text}\n\n질문: {question}"},
        ],
        temperature=0.3,
    ).choices[0].message.content.strip()


def judge_faithfulness(response: str, contexts: list[str]) -> float:
    context_text = "\n\n---\n\n".join(contexts[:3])
    raw = llm.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": JUDGE_PROMPT.format(
            context=context_text, response=response,
        )}],
        temperature=0,
    ).choices[0].message.content.strip()
    try:
        return float(raw)
    except ValueError:
        return 0.5  # 파싱 실패 시 중립


print("Answer Faithfulness 평가 시작...")
for i, r in enumerate(results):
    answer   = generate_answer(r["question"], r["contexts"])
    faith    = judge_faithfulness(answer, r["contexts"])
    r["llm_answer"]       = answer
    r["faithfulness"]     = faith
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(results)} 완료")

avg_faith = sum(r["faithfulness"] for r in results) / len(results)
print(f"\n전체 Answer Faithfulness: {avg_faith:.3f}")

Answer Faithfulness 평가 시작...
  10/48 완료
  20/48 완료
  30/48 완료
  40/48 완료

전체 Answer Faithfulness: 1.000


## 4. 카테고리별 종합 성능

In [8]:
import pandas as pd

cat_stats: dict[str, dict] = defaultdict(lambda: {"recall": [], "faithfulness": []})
for r in results:
    cat_stats[r["category"]]["recall"].append(r["context_recall"])
    cat_stats[r["category"]]["faithfulness"].append(r["faithfulness"])

rows = []
for cat, v in sorted(cat_stats.items()):
    n   = len(v["recall"])
    avg_r = sum(v["recall"]) / n
    avg_f = sum(v["faithfulness"]) / n
    rows.append({
        "카테고리":       cat,
        "샘플수":         n,
        "Context Recall": round(avg_r, 3),
        "Faithfulness":   round(avg_f, 3),
    })

rows.append({
    "카테고리":       "[전체 평균]",
    "샘플수":         len(results),
    "Context Recall": round(avg_recall, 3),
    "Faithfulness":   round(avg_faith, 3),
})

pd.DataFrame(rows).set_index("카테고리")

,샘플수,Context Recall,Faithfulness
카테고리,,,
건강 및 질병,10,0.980,1.0
사육 및 관리,10,0.950,1.0
여행 및 이동,8,1.000,1.0
영양 및 식단,10,1.000,1.0
행동 및 심리,10,0.980,1.0
[전체 평균],48,0.981,1.0


## 5. 오답 케이스 분석

Context Recall < 0.5 또는 Faithfulness < 0.5 인 케이스 확인

In [9]:
weak = [
    r for r in results
    if r["context_recall"] < 0.5 or r["faithfulness"] < 0.5
]
print(f"취약 케이스: {len(weak)}/{len(results)}개\n")

for r in weak:
    print(f"[{r['category']} / {r['species']}]")
    print(f"  Q: {r['question']}")
    print(f"  Gold A: {r['answer'][:80]}...")
    print(f"  키워드: {r['keywords']}")
    print(f"  Context Recall: {r['context_recall']}  |  Faithfulness: {r['faithfulness']}")
    print(f"  LLM 답변: {r['llm_answer'][:100]}...")
    print()

취약 케이스: 0/48개



In [10]:
# 결과 저장 (재실행 없이 분석 재사용)
save_path = HERE / "data" / "rag_eval_results.json"
save_data = [
    {k: v for k, v in r.items() if k != "contexts"}  # contexts는 용량 절약을 위해 제외
    for r in results
]
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(save_data, f, ensure_ascii=False, indent=2)
print(f"결과 저장: {save_path}")

결과 저장: /Users/leemdo/Workspaces/SKN22-Final-2Team-WEB/notebooks/langgraph/data/rag_eval_results.json
